In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
from arch import arch_model
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from curl_cffi import requests

session = requests.Session(impersonate='chrome')

class StockData:
    def __init__(self, tickers, start_date, end_date):
        self.tickers = tickers
        self.start_date = start_date
        self.end_date = end_date
        self.returns_scaled = None

    def fetch_data(self):
        data = yf.download(self.tickers, start=self.start_date, end=self.end_date, session = session)['Close']
        returns = np.log(data / data.shift(1)).dropna()
        self.returns_scaled = returns * 100  # Scale the returns


class GARCHModel:
    def __init__(self, returns_scaled):
        self.returns_scaled = returns_scaled
        self.sigmas = pd.DataFrame()

    def fit_models(self):
        for ticker in self.returns_scaled.columns:
            model = arch_model(self.returns_scaled[ticker], vol='Garch', p=1, q=1)
            result = model.fit(disp="off")
            self.sigmas[ticker] = result.conditional_volatility
        self.sigmas.index = self.returns_scaled.index


class DCCGARCH:
    def __init__(self, sigmas, returns_scaled, tickers):
        self.sigmas = sigmas
        self.returns_scaled = returns_scaled
        self.tickers = tickers
        self.dcc_correlations = None

    def calculate_dynamic_correlation(self, window=30):
        correlations = []
        for i in range(window, len(self.returns_scaled)):
            subset_returns = self.returns_scaled.iloc[i - window:i]
            subset_sigmas = self.sigmas.iloc[i - window:i]
            standardized_returns = subset_returns / subset_sigmas
            corr_matrix = np.corrcoef(standardized_returns.T)
            correlations.append(corr_matrix[0, 1])
        self.dcc_correlations = np.array(correlations)


class PlotAll:
    def __init__(self, returns_scaled, sigmas, dcc_correlations, tickers):
        self.returns_scaled = returns_scaled
        self.sigmas = sigmas
        self.dcc_correlations = dcc_correlations
        self.tickers = tickers

    def plot(self):
        fig = make_subplots(specs=[[{"secondary_y": True}]])
        # Plot returns
        for ticker in self.tickers:
            fig.add_trace(go.Scatter(x=self.returns_scaled.index, y=self.returns_scaled[ticker], mode='lines', name=f"{ticker} Returns"))
        # Plot volatilities
        for ticker in self.tickers:
            fig.add_trace(go.Scatter(x=self.sigmas.index, y=self.sigmas[ticker], mode='lines', name=f"{ticker} GARCH Volatility", line=dict(dash='dash')))
        # Plot DCC correlation on secondary y-axis
        fig.add_trace(go.Scatter(x=self.returns_scaled.index[-len(self.dcc_correlations):], y=self.dcc_correlations, mode='lines', name='DCC Correlation', line=dict(color='pink')), secondary_y=True)

        fig.update_layout(
            title='DCC-GARCH Analysis',
            xaxis_title='Date',
            yaxis_title='Returns / Volatility',
            yaxis2_title='DCC Correlation',
            height=600, width=700
        )
        fig.show()


def run_analysis():
    tickers = ['AAPL', 'MSFT']
    start_date = "2020-01-01"
    end_date = "2024-06-01"

    # Step 1: Fetch stock data (Scaled Log Returns)
    stock_data = StockData(tickers, start_date, end_date)
    stock_data.fetch_data()

    # Step 2: Fit GARCH models and extract volatilities
    garch_model = GARCHModel(stock_data.returns_scaled)
    garch_model.fit_models()

    # Step 3: Calculate DCC-GARCH dynamic correlations
    dcc_model = DCCGARCH(garch_model.sigmas, stock_data.returns_scaled, tickers)
    dcc_model.calculate_dynamic_correlation()

    # Step 4: Plot all information on a single plot
    plot_all = PlotAll(stock_data.returns_scaled, garch_model.sigmas, dcc_model.dcc_correlations, tickers)
    plot_all.plot()

# Run the analysis
if __name__ == "__main__":
    run_analysis()

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  2 of 2 completed
